In [1]:
import pandas as pd

In [46]:
df = pd.read_csv("../data/water_potability.csv")

In [47]:
df

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0
...,...,...,...,...,...,...,...,...,...,...
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1
3272,7.808856,193.553212,17329.802160,8.061362,NaN,392.449580,19.903225,NaN,2.798243,1
3273,9.419510,175.762646,33155.578218,7.350233,NaN,432.044783,11.039070,69.845400,3.298875,1
3274,5.126763,230.603758,11983.869376,6.303357,NaN,402.883113,11.168946,77.488213,4.708658,1


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ph               2785 non-null   float64
 1   Hardness         3276 non-null   float64
 2   Solids           3276 non-null   float64
 3   Chloramines      3276 non-null   float64
 4   Sulfate          2495 non-null   float64
 5   Conductivity     3276 non-null   float64
 6   Organic_carbon   3276 non-null   float64
 7   Trihalomethanes  3114 non-null   float64
 8   Turbidity        3276 non-null   float64
 9   Potability       3276 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 256.1 KB


In [49]:
df.isnull().sum()

ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64

In [50]:
df.dtypes

ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object

In [51]:
import seaborn as sns

In [52]:
df['Potability'].value_counts(normalize=True)

Potability
0    0.60989
1    0.39011
Name: proportion, dtype: float64

In [53]:
df.describe()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
count,2785.000000,3276.000000,3276.000000,3276.000000,2495.000000,3276.000000,3276.000000,3114.000000,3276.000000,3276.000000
mean,7.080795,196.369496,22014.092526,7.122277,333.775777,426.205111,14.284970,66.396293,3.966786,0.390110
std,1.594320,32.879761,8768.570828,1.583085,41.416840,80.824064,3.308162,16.175008,0.780382,0.487849
min,0.000000,47.432000,320.942611,0.352000,129.000000,181.483754,2.200000,0.738000,1.450000,0.000000
25%,6.093092,176.850538,15666.690297,6.127421,307.699498,365.734414,12.065801,55.844536,3.439711,0.000000
50%,7.036752,196.967627,20927.833607,7.130299,333.073546,421.884968,14.218338,66.622485,3.955028,0.000000
75%,8.062066,216.667456,27332.762127,8.114887,359.950170,481.792304,16.557652,77.337473,4.500320,1.000000
max,14.000000,323.124000,61227.196008,13.127000,481.030642,753.342620,28.300000,124.000000,6.739000,1.000000


In [54]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Potability'])
y = df['Potability']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((2620, 9), (656, 9), (2620,), (656,))

In [55]:
num_cols = X_train.columns

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [13]:
from sklearn.impute import KNNImputer

num_imputer = KNNImputer(n_neighbors=5)

In [14]:
numeric_pipeline = Pipeline(steps=[("imputer",num_imputer),("scaler",StandardScaler())])
numeric_pipeline

,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,n_neighbors,5
,weights,'uniform'
,metric,'nan_euclidean'
,copy,True
,add_indicator,False
,keep_empty_features,False


In [15]:
preprocessor = ColumnTransformer(transformers=[("num",numeric_pipeline,num_cols)])
preprocessor

,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,n_neighbors,5
,weights,'uniform'


In [16]:
from sklearn.linear_model import LogisticRegression

model_logistic = LogisticRegression(penalty="l2",class_weight='balanced',solver="liblinear",random_state=42)

pipeline_logistic = Pipeline(steps=[("preprocessing",preprocessor),("model",model_logistic)])
pipeline_logistic

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
X_test

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
664,NaN,188.743562,19037.462638,6.034236,NaN,388.065857,15.149068,78.499418,2.723651
2530,6.172517,187.787017,10498.635154,7.722561,322.521035,467.090425,21.233978,68.163642,4.129985
2870,NaN,176.386651,26003.163265,7.809251,358.429774,336.142005,14.447961,90.224844,4.410674
1045,6.369112,235.340943,34456.801132,9.170940,295.350524,357.417285,9.213268,59.280269,2.275903
95,6.140878,197.876090,26687.874483,7.587196,329.231853,548.072761,15.836330,41.263648,5.359460
...,...,...,...,...,...,...,...,...,...
322,7.798454,188.394942,32704.569286,11.078872,258.191184,507.178688,18.272439,85.177662,4.107267
2042,7.669013,205.595635,11579.441693,4.263279,356.136518,407.721613,10.829045,83.243808,4.589513
1696,7.290089,205.213105,26115.616326,5.137891,357.794088,402.874799,7.960478,63.698514,4.700618
1259,NaN,164.094744,9344.653322,5.959309,349.019434,367.389811,13.937968,88.771969,4.559337


In [18]:
from sklearn.metrics import f1_score, classification_report
pipeline_logistic.fit(X_train, y_train)
y_pred_log = pipeline_logistic.predict(X_test)
print("F1 Score: ",f1_score(y_test,y_pred_log))
print("Classification report: ")
print(classification_report(y_test,y_pred_log))

F1 Score:  0.47440273037542663
Classification report: 
              precision    recall  f1-score   support

           0       0.64      0.52      0.58       400
           1       0.42      0.54      0.47       256

    accuracy                           0.53       656
   macro avg       0.53      0.53      0.53       656
weighted avg       0.56      0.53      0.54       656



In [19]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=300,random_state=42,class_weight="balanced")

pipeline = Pipeline(steps=[("preprocessing",preprocessor),("model",model)])
pipeline

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
print("F1 Score: ",f1_score(y_test,y_pred))
print("Classification report: ")
print(classification_report(y_test,y_pred))

F1 Score:  0.38904109589041097
Classification report: 
              precision    recall  f1-score   support

           0       0.66      0.91      0.76       400
           1       0.65      0.28      0.39       256

    accuracy                           0.66       656
   macro avg       0.66      0.59      0.58       656
weighted avg       0.66      0.66      0.62       656



In [21]:
pipeline.predict(X_train.sample(6))

array([0, 0, 1, 0, 0, 0])

In [22]:
df.sample(1)

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
2820,NaN,168.409525,26148.537873,9.03044,424.302122,445.345875,10.674582,51.876955,3.893327,1


In [23]:
importances = model.feature_importances_
importances

array([0.13455792, 0.11774768, 0.10940089, 0.11459137, 0.13701502,
       0.09951894, 0.09583595, 0.09493453, 0.09639769])

In [24]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 5, 10],
    "model__penalty": ["l2"],
    "model__class_weight": [None, "balanced"],
    "model__max_iter": [300, 500, 800]
}

In [25]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=pipeline_logistic,
    param_grid=param_grid,
    scoring="f1_weighted",       
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [26]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,Pipeline(step...liblinear'))])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__class_weight': [None, 'balanced'], 'model__max_iter': [300, 500, ...], 'model__penalty': ['l2']}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [27]:
print("Best parameters:", grid_search.best_params_)
print("Best CV F1-score:", grid_search.best_score_)

Best parameters: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 300, 'model__penalty': 'l2'}
Best CV F1-score: 0.49105620709296316


In [28]:
param_grid = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

In [29]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_weighted",   
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [30]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [31]:
print("Best parameters:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

Best parameters: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best CV F1: 0.6482838409603782


In [32]:
import pandas as pd
best_model = grid_search.best_estimator_
rf_model = best_model.named_steps["model"]
importance = rf_model.feature_importances_
pd.Series(importance, index=X.columns).sort_values(ascending=False)

Sulfate            0.174908
ph                 0.159566
Hardness           0.124025
Chloramines        0.120388
Solids             0.107892
Conductivity       0.081949
Organic_carbon     0.079849
Turbidity          0.076524
Trihalomethanes    0.074898
dtype: float64

In [44]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd

class FeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        X["ph_sulfate"] = X["ph"] * X["Sulfate"]
        X["hardness_sulfate"] = X["Hardness"] * X["Sulfate"]
        X["ph_hardness"] = X["ph"] * X["Hardness"]

        X["Solids_log"] = np.log1p(X["Solids"])

        return X

In [45]:
from sklearn.pipeline import Pipeline

pipeline_RF = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier())
])
pipeline_RF

,steps,"[('feature_engineering', ...), ('preprocessing', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [56]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Potability'])
y = df['Potability']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((2620, 9), (656, 9), (2620,), (656,))

In [66]:
param_grid = {
    "model__n_estimators": [200, 300, 400],          # 3
    "model__max_depth": [None, 5, 10, 15],           # 4
    "model__min_samples_split": [2, 5, 10],          # 3
    "model__min_samples_leaf": [1, 2, 4],            # 3
    "model__max_features": ["sqrt", "log2"]          # 2
}

In [71]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1,
    verbose=2
)

In [72]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [73]:
print("Best parameters:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

Best parameters: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best CV F1: 0.6482838409603782


In [ ]:
import joblib
best_pipeline = grid_search.best_estimator_
joblib.dump(best_pipeline, "water_potability_pipeline.pkl")

['../models/water_potability_pipeline.pkl']

In [ ]:
model = joblib.load("water_potability_pipeline.pkl")

pred = model.predict(X_test)

print(pred[:10])

[0 0 0 0 0 0 0 0 0 1]


In [ ]:
import joblib
import pandas as pd

model = joblib.load("water_potability_pipeline.pkl")

sample = pd.DataFrame([{
    "ph":7,
    "Hardness":200,
    "Solids":10000,
    "Chloramines":7,
    "Sulfate":350,
    "Conductivity":500,
    "Organic_carbon":10,
    "Trihalomethanes":80,
    "Turbidity":4
}])

print(model.predict(sample))

[0]
